In [1]:
import os
from dotenv import load_dotenv  # Carrega variáveis de ambiente do arquivo .env
import dspy  # Framework para otimização de prompts com Language Models

load_dotenv()  # Lê variáveis do arquivo .env

True

## Setup - Configuração do Modelo

Carregamos as variáveis de ambiente do arquivo `.env` na raiz do projeto. Isso evita hardcoding de credenciais no código.

In [2]:
lm = dspy.LM(
    "openai/gpt-5-mini",  # Modelo OpenAI GPT-5 Mini
    api_key=os.getenv("OPENAI_API_KEY"),  # API key carregada da variável de ambiente
)

# Configura o modelo padrão para todas as operações DSPy
dspy.configure(lm=lm)

## ChainOfThought

`ChainOfThought` é um módulo que solicita ao modelo para exibir seu raciocínio antes de fornecer a resposta final. Útil para:
- Tarefas que requerem múltiplas etapas
- Problemas matemáticos complexos
- Decisões que precisam de justificativa

Retorna dois campos: `reasoning` (pensamento passo-a-passo) e `resposta` (resultado final).

In [3]:
class CalculadoraMatematica(dspy.Signature):
    """Resolva o problema matemático."""

    # Campo de entrada: pergunta matemática
    pergunta: str = dspy.InputField(desc="Pergunta matemática a ser resolvida")

    # Campo de saída: resposta numérica final
    resposta: str = dspy.OutputField(desc="Resposta numérica final do problema")

## Criar Módulo ChainOfThought

Em vez de `dspy.Predict()`, usamos `dspy.ChainOfThought()`. DSPy adiciona automaticamente um campo `reasoning` que captura o pensamento intermediário do modelo.

In [4]:
# Cria um ChainOfThought que usa a Signature classe
# DSPy adiciona automaticamente o campo 'reasoning' para raciocínio passo-a-passo
calculadora = dspy.ChainOfThought(CalculadoraMatematica)

## Executar ChainOfThought

Passamos a pergunta e o modelo gera resposta com raciocínio intermediário.

In [5]:
resultado = calculadora(
    pergunta="Qual é o valor de 15% de 840 somado a 27% de 500?"  # Pergunta que requer múltiplos cálculos
)

resultado  # Retorna Prediction com .reasoning e .resposta

Prediction(
    reasoning='Calculo 15% de 840: (15/100) × 840 = 0,15 × 840 = 126.\nCalculo 27% de 500: (27/100) × 500 = 0,27 × 500 = 135.\nSomando: 126 + 135 = 261.',
    resposta='261'
)

## Ver Raciocínio

ChainOfThought retorna o campo `reasoning` com o pensamento passo-a-passo do modelo.

In [6]:
print("=" * 70)
print("RACIOCÍNIO")
print("=" * 70)
print(resultado.reasoning)  # Mostra o pensamento passo-a-passo

RACIOCÍNIO
Calculo 15% de 840: (15/100) × 840 = 0,15 × 840 = 126.
Calculo 27% de 500: (27/100) × 500 = 0,27 × 500 = 135.
Somando: 126 + 135 = 261.


## Ver Resposta Final

Extrai apenas a resposta numérica final.

In [7]:
print("\n" + "=" * 70)
print("RESPOSTA FINAL")
print("=" * 70)
print(resultado.resposta)  # Mostra apenas o resultado final


RESPOSTA FINAL
261


## Inspecionando o Histórico

Vemos como o ChainOfThought estrutura o prompt de forma diferente, pedindo explicitamente pelo raciocínio.

In [8]:
# Mostra última chamada ao modelo (n=1 significa 1 última chamada)
dspy.inspect_history(n=1)





[2026-09-05T22:10:08.071203]

System message:

Your input fields are:
1. `pergunta` (str): Pergunta matemática a ser resolvida
Your output fields are:
1. `reasoning` (str): 
2. `resposta` (str): Resposta numérica final do problema
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## pergunta ## ]]
{pergunta}

[[ ## reasoning ## ]]
{reasoning}

[[ ## resposta ## ]]
{resposta}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Resolva o problema matemático.


User message:

[[ ## pergunta ## ]]
Qual é o valor de 15% de 840 somado a 27% de 500?

Respond with the corresponding output fields, starting with the field `[[ ## reasoning ## ]]`, then `[[ ## resposta ## ]]`, and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## reasoning ## ]]
Calculo 15% de 840: (15/100) × 840 = 0,15 × 840 = 126.
Calculo 27% de 500: (27/100) × 500 = 0,27 × 500 = 135.
Somando: 126 + 135 = 261.

[[ ## re

## Comparação: Predict vs ChainOfThought

Para comparação, vamos executar a mesma pergunta com um Predict simples (sem raciocínio passo-a-passo).

In [9]:
# Cria um Predict simples (sem raciocínio)
calculadora_simples = dspy.Predict(CalculadoraMatematica)

resultado_simples = calculadora_simples(
    pergunta="Qual é o valor de 15% de 840 somado a 27% de 500?"
)

print("\n" + "=" * 70)
print("PREDICT (SEM RACIOCÍNIO)")
print("=" * 70)
print(resultado_simples.resposta)


PREDICT (SEM RACIOCÍNIO)
261


## Inspecionando o Histórico

Vemos como o ChainOfThought estrutura o prompt de forma diferente, pedindo explicitamente pelo raciocínio.

In [10]:
# Mostra última chamada ao modelo (n=1 significa 1 última chamada)
dspy.inspect_history(n=1)





[2026-09-05T22:10:08.167029]

System message:

Your input fields are:
1. `pergunta` (str): Pergunta matemática a ser resolvida
Your output fields are:
1. `resposta` (str): Resposta numérica final do problema
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## pergunta ## ]]
{pergunta}

[[ ## resposta ## ]]
{resposta}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Resolva o problema matemático.


User message:

[[ ## pergunta ## ]]
Qual é o valor de 15% de 840 somado a 27% de 500?

Respond with the corresponding output fields, starting with the field `[[ ## resposta ## ]]`, and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## resposta ## ]]
261

[[ ## completed ## ]]





